In [ ]:
%pip install transformers torch peft textblob

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, multilabel_confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, ElectraTokenizer, ElectraForSequenceClassification
import torch
from torch.optim import AdamW
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType
import copy

MAX_LENGTH  = 64
BATCH_SIZE  = 16
EPOCHS      = 10
LR          = 5e-5
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, num_classes, max_length=512):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = torch.zeros(len(labels), num_classes)
        for i, label_list in enumerate(labels):
            if isinstance(label_list, int):
                label_list = [label_list]
            self.labels[i, label_list] = 1.0

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx]
        }

def eval_model(model, loader):
    all_preds = []
    all_labels = []
    model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            probs   = torch.sigmoid(outputs.logits)
            preds   = (probs > 0.5).int()
            all_preds.append(preds.cpu())
            all_labels.append(batch["labels"].int().cpu())
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    results_df = pd.DataFrame(
        np.hstack([all_labels, all_preds]),
        columns=label_cols + pred_cols
    )
    return results_df

In [2]:
def eval_and_print_results(model, model_name, label_mappings, label_cols, pred_cols):
    loader = f'{model_name.split('_'[0]).lower()}_val_loader'
    results_df = eval_model(model, loader)
    plot_confusion_heatmap_grid(results_df, model_name, label_mappings)
    plot_confusion_heatmap(results_df, model_name, label_cols, pred_cols)
    return results_df

def extract_labels(df, label_mappings):
    """
    Got this from Claude because I had only worked with Class n style labels and needed
    to shoehorn the actual class labels in once I found them
    Extracts y_true and y_pred from a df with one-hot Truth_/Predicted_ columns.
    For multi-label rows, each (true, predicted) pair is expanded into separate entries.
    """
    labels = list(label_mappings.values())
    truth_cols = {label: f'Truth_{label}' for label in labels}
    pred_cols  = {label: f'Predicted_{label}' for label in labels}

    y_true, y_pred = [], []

    for _, row in df.iterrows():
        true_labels = [label for label in labels if row.get(truth_cols[label], 0) == 1]
        pred_labels = [label for label in labels if row.get(pred_cols[label],  0) == 1]

        for t in (true_labels or ['neutral']):
            for p in (pred_labels or ['neutral']):
                y_true.append(t)
                y_pred.append(p)

    return y_true, y_pred


def plot_confusion_heatmap(df, model_name, label_cols, pred_cols, label_mappings=None):
    y_true = df[label_cols].values
    y_pred = df[pred_cols].values
    mcm = multilabel_confusion_matrix(y_true, y_pred)

    # --- Heatmap ---
    data = np.array([[m[1,1], m[0,1], m[1,0], m[0,0]] for m in mcm])
    row_labels = (
        [label_mappings[label_cols[i]] for i in range(len(label_cols))]
        if label_mappings else
        [f'Class {label_cols[i]}' for i in range(len(label_cols))]
    )
    sns.heatmap(
        data,
        annot=True, fmt='d', cmap='Blues',
        xticklabels=['TP', 'FP', 'FN', 'TN'],
        yticklabels=row_labels
    )
    plt.title(f'{model_name} — per-class confusion matrix')
    plt.tight_layout()
    plt.show()

    # --- Classification report ---
    target_names = row_labels
    report = classification_report(
        y_true, y_pred,
        target_names=target_names,
        zero_division=0,
        output_dict=True
    )
    report_df = pd.DataFrame(report).T
    print(f"\n{model_name} — classification report")
    print(report_df.to_string(float_format=lambda x: f"{x:.4f}"))
    
def plot_confusion_heatmap_grid(df, model_name, label_mappings):
    labels = list(label_mappings.values())
    y_true, y_pred = extract_labels(df, label_mappings)

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_normalized = np.nan_to_num(cm_normalized)

    n = len(labels)
    fig, ax = plt.subplots(figsize=(14, 12))

    sns.heatmap(
        cm_normalized,
        ax=ax,
        cmap='Reds',
        xticklabels=labels,
        yticklabels=labels,
        vmin=0.0,
        vmax=1.0,
        linewidths=0.3,
        linecolor='white',
        cbar_kws={'label': 'Proportion of Predictions', 'shrink': 0.6}
    )
    ax.set_title(f'{model_name} — Multilabel Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Predicted Label', fontsize=12, labelpad=10)
    ax.set_ylabel('True Label', fontsize=12, labelpad=10)
    ax.tick_params(axis='x', rotation=90, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)

    plt.tight_layout()
    plt.savefig('confusion_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def apply_lora(model, target_modules):
    config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=8,                  
        lora_alpha=16,        
        lora_dropout=0.1,
        target_modules= target_modules
    )
    return get_peft_model(model, config)
    
def train(model, train_loader, val_loader, epochs=3, lr=2e-4, patience=3):
    optimizer = AdamW(model.parameters(), lr=lr)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    best_val_loss = float('inf')
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            del outputs, loss                          
        torch.cuda.empty_cache()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                val_loss += outputs.loss.item()
                del outputs                            
        torch.cuda.empty_cache()

        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch+1} — val loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(        
                {k: v.cpu() for k, v in model.state_dict().items()}
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                break

    model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    return model

In [ ]:
def save_model(model, tokenizer, save_path):
    import os
    os.makedirs(save_path, exist_ok=True)
    model.save_pretrained(save_path) 
    tokenizer.save_pretrained(save_path)
    print(f"Model saved to {save_path}")

def load_model(save_path, base_model_name, num_labels):
    from peft import PeftModel
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    base_model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        num_labels=num_labels
    )
    model = PeftModel.from_pretrained(base_model, save_path) 
    tokenizer = AutoTokenizer.from_pretrained(save_path)
    return model, tokenizer

In [ ]:
# Define Data & Splits

splits = {'train': 'simplified/train-00000-of-00001.parquet', 'validation': 'simplified/validation-00000-of-00001.parquet', 'test': 'simplified/test-00000-of-00001.parquet'}
df_train = pd.read_parquet("hf://datasets/google-research-datasets/go_emotions/" + splits["train"])
# df_train = df_train.iloc[:10,:]
df_val = pd.read_parquet("hf://datasets/google-research-datasets/go_emotions/" + splits["validation"])
# df_val = df_val.iloc[:10,:]
df_test = pd.read_parquet("hf://datasets/google-research-datasets/go_emotions/" + splits["test"])
# df_test = df_test.iloc[:10,:]
df_train.head()

In [ ]:
# Make the data splits for each model

#----------------
# Electra Data
#----------------
electra_tokenizer = ElectraTokenizer.from_pretrained('google/electra-base-discriminator')

electra_train_dataset = TextDataset(df_train["text"].tolist(), df_train["labels"].tolist(), electra_tokenizer, NUM_LABELS)
electra_val_dataset   = TextDataset(df_val["text"].tolist(),   df_val["labels"].tolist(),   electra_tokenizer, NUM_LABELS)
electra_test_dataset   = TextDataset(df_test["text"].tolist(),   df_test["labels"].tolist(),   electra_tokenizer, NUM_LABELS)

electra_train_loader  = DataLoader(electra_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
electra_val_loader    = DataLoader(electra_val_dataset,   batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)
electra_test_loader    = DataLoader(electra_test_dataset,   batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)

#----------------
# DistilBERT Data
#----------------
distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

distilbert_train_dataset = TextDataset(df_train["text"].tolist(), df_train["labels"].tolist(), distilbert_tokenizer, NUM_LABELS)
distilbert_val_dataset   = TextDataset(df_val["text"].tolist(),   df_val["labels"].tolist(),   distilbert_tokenizer, NUM_LABELS)
distilbert_test_dataset   = TextDataset(df_test["text"].tolist(),   df_test["labels"].tolist(),   distilbert_tokenizer, NUM_LABELS)

distilbert_train_loader  = DataLoader(distilbert_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
distilbert_val_loader    = DataLoader(distilbert_val_dataset,   batch_size=BATCH_SIZE)
distilbert_test_loader    = DataLoader(distilbert_test_dataset,   batch_size=BATCH_SIZE)

In [ ]:
#----------------
# Electra Base Model
#----------------

electra_base_model = ElectraForSequenceClassification.from_pretrained(
    'google/electra-base-discriminator',
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

In [ ]:
#----------------
# DistilBERT Base Model
#----------------

distilbert_base_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

In [ ]:
# Results of base models
electra_results_df = eval_and_print_results(electra_base_model, 'Electra', label_mappings, label_cols, pred_cols)
distilbert_results_df = eval_and_print_results(distilbert_base_model, 'DistilBERT', label_mappings, label_cols, pred_cols)

In [ ]:
# -------------------------
# Finetuning Starts Here
#--------------------------
# for name, module in electra_base_model.named_modules():
#     print(name)
# for name, module in distilbert_base_model.named_modules():
#     print(name)
# I ran these and was like what have I gotten into...

distilbert_lora_model = apply_lora(distilbert_base_model, ["q_lin", "v_lin"]) # Totally just asked Claude what to put here. No clue
electra_lora_model = apply_lora(electra_base_model, ["query", "value"]) # Here too

In [ ]:
#----------------
# Train finetuned models
#----------------

# I didn't run this cell in this notebook because what you see 
# in this notebook is a distillation of days of fits and false starts
# If you really need it I have the other one. It works. Really.

print('Training Electra')
electra_fine_model = train(electra_lora_model,electra_train_loader, electra_val_loader, epochs=EPOCHS, lr=LR)
save_model(electra_fine_model,  electra_tokenizer, "./Electra_5e-5")

print('Training DistilBERT')
distilbert_fine_model = train(distilbert_lora_model, distilbert_train_loader, distilbert_val_loader, epochs=EPOCHS, lr=LR)
save_model(distilbert_fine_model,  distilbert_tokenizer, "./DistilBERT_5e-5")

In [ ]:
electra_fine_model3,  electra_tokenizer3 = load_model(
    save_path="./Electra_5e-5",
    base_model_name="google/electra-base-discriminator",
    num_labels=28
)
distilbert_fine_model3,  distilbert_tokenizer3 = load_model(
    save_path="./DistilBERT_5e-5",
    base_model_name="distilbert-base-uncased",
    num_labels=28
)

In [ ]:
#----------------
# Results of finetuned models
#----------------

electra_results_fine_df = eval_and_print_results(electra_fine_model3, 'Electra', label_mappings, label_cols, pred_cols)
distilbert_results_fine_df = eval_and_print_results(distilbert_fine_model3, 'DistilBERT', label_mappings, label_cols, pred_cols)